In [6]:
from smolagents import CodeAgent, LiteLLMModel
from dotenv import load_dotenv
from agents.agent_runner import StepRunner, start_stepper, step_once, is_finished, finish
import os
import time
load_dotenv()
#model = LiteLLMModel(model_id="gpt-5-mini")
model = LiteLLMModel(model_id="gemini/gemini-2.5-flash", api_key=os.getenv("GOOGLE_API_KEY"))

agent = CodeAgent(tools=[], model=model, verbosity_level=2)
print("Agent ready ✅")

Agent ready ✅


In [7]:
# Demo: run step-by-step and finish
runner = start_stepper(agent, "What is 2**10?", reset=True)

# Advance until completion using the helper
while not is_finished(runner):
    step_once(runner)
    time.sleep(5)  
finish(runner)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is 2**10?                                                                                                  │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

[Step 1: Duration 9.92 seconds| Input tokens: 2,063 | Output tokens: 47]

Exception ignored in: <generator object MultiStepAgent._run_stream at 0x7f8f78daf610>
Traceback (most recent call last):
  File "/tmp/ipykernel_10069/4082456430.py", line 2, in <module>
RuntimeError: generator ignored GeneratorExit


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
<code>                                                                                                             
result = 2**10                                                                                                     
final_answer(result)                                                                                               
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  result = 2**10                                                                                                   
  final_answer(result)                                                                                             
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: 1024

[Step 1: Duration 11.59 seconds| Input tokens: 2,063 | Output tokens: 62]

In [ ]:
# Minimal vision agent: answer "what's on the image?" for a given URL
from io import BytesIO
import os
import requests
from PIL import Image
from dotenv import load_dotenv
from smolagents import CodeAgent, LiteLLMModel

# Using a reliable, direct image URL from the documentation to ensure it works.
IMAGE_URL = "https://gratisography.com/wp-content/uploads/2024/11/gratisography-augmented-reality-1170x780.jpg"

# 1. Load the image from the direct URL
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/118.0.0.0 Safari/537.36"
}
try:
    response = requests.get(IMAGE_URL, headers=headers, timeout=30)
    response.raise_for_status()  # Ensure we got a valid response
    image = Image.open(BytesIO(response.content)).convert("RGB")
    images = [image]
    print("Image loaded successfully.")
except Exception as e:
    print(f"Failed to load image: {e}")
    images = []

if images:
    load_dotenv()
    model_id = os.getenv("VISION_MODEL_ID", "gemini/gemini-2.5-flash")
    api_key_env = os.getenv("VISION_API_KEY_ENV", "GOOGLE_API_KEY")
    api_key = os.getenv(api_key_env)

    if not api_key:
        raise RuntimeError(
            f"Missing API key in environment variable '{api_key_env}'. "
            "Please set it in a .env file or your system environment."
        )

    model = LiteLLMModel(model_id=model_id, api_key=api_key)
    agent = CodeAgent(tools=[], model=model, max_steps=5, verbosity_level=1)

    # 3. Ask the question with the image
    question = "What's on the image? Describe the main subject concisely."
    final_answer = agent.run(question, images=images)
    print("\nFinal answer:", final_answer)


Image loaded successfully.


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What's on the image? Describe the main subject concisely.                                                       │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("A person in a hooded sweatshirt wearing a device with glowing circular 'eyes'.")                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: A person in a hooded sweatshirt wearing a device with glowing circular 'eyes'.

[Step 1: Duration 1.47 seconds| Input tokens: 2,326 | Output tokens: 66]


Final answer: A person in a hooded sweatshirt wearing a device with glowing circular 'eyes'.


In [18]:
# New: run the vision agent step-by-step
if images:
    # The StepRunner from agent_runner.py can take kwargs, which are passed to agent.run()
    # This is how we can pass the `images` argument.
    runner = start_stepper(agent, "What is on the image? Describe it.", reset=True, images=images)

    # Advance until completion using the helper
    while not is_finished(runner):
        output = step_once(runner)
        print(f"Step yielded: {type(output).__name__}")
        # Optional: print the details of the output
        # print(output)
        time.sleep(1)
    
    finish(runner)
    print("Step-by-step vision task finished.")


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ What is on the image? Describe it.                                                                              │
│                                                                                                                 │
╰─ LiteLLMModel - gemini/gemini-2.5-flash ────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Step yielded: ToolCall


─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("The image depicts a figure wearing a greyish-blue hoodie. Their face is obscured by a dark,        
  mask-like device featuring two glowing orange-rimmed circular 'eyes.' The figure's hands are raised, seemingly   
  adjusting the hood over their head. The background is a plain, dark teal color.")                                
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Step yielded: ActionOutput


Final answer: The image depicts a figure wearing a greyish-blue hoodie. Their face is obscured by a dark, mask-like
device featuring two glowing orange-rimmed circular 'eyes.' The figure's hands are raised, seemingly adjusting the 
hood over their head. The background is a plain, dark teal color.

[Step 1: Duration 4.33 seconds| Input tokens: 2,322 | Output tokens: 248]

Step yielded: ActionStep
Step yielded: FinalAnswerStep
Step yielded: FinalAnswerStep
Step yielded: NoneType
Step yielded: NoneType
Step-by-step vision task finished.
Step-by-step vision task finished.
